In [2]:
import pandas as pd
import pyreadr

result = pyreadr.read_r("../data/raw/BRinf.rda")
print(result.keys())

odict_keys(['BRinf'])


In [3]:
df = result['BRinf'].copy()

df.head()

,Brazil CPI IPCA MoM,FGV Brazil General Prices IGP-,FGV Brazil General Prices IGP-,FGV Brazil General Prices IGP-,FGV Brazil General Prices IGP-,Brazil CPI IPCA Median Market,Brazil Total Electricity Consu,Brazil Industrial Electricity,BofA Merrill Lynch Economic Co,CNI Brazil Manufacture Industr,...,X5t.12,X5t.13,fmed2,fmean,fmean2,fdp,lfmed2,lfmean,lfmean2,lfdp
2003-01-31,2.25,2.33,2.17,5.816,2.29,11.56,25422.0,-359.0,-1.47,0.0000,...,0.80,0.68,4.0000,1.96,3.8416,0.18,1.3225,1.15,1.3225,0.24
2003-02-28,1.57,2.28,1.59,4.351,2.42,10.95,25163.0,-303.0,-1.21,80.5999,...,0.68,0.48,2.5600,1.60,2.5600,0.26,0.7569,0.91,0.8281,0.27
2003-03-31,1.23,1.53,1.66,4.593,1.58,9.58,24960.0,115.0,-0.35,-0.3000,...,0.55,0.50,0.8836,0.93,0.8649,0.13,0.4900,0.73,0.5329,0.17
2003-04-30,0.97,0.92,0.41,1.173,1.24,8.88,25284.0,93.0,-0.15,-0.5000,...,0.45,0.78,1.0000,0.97,0.9409,0.14,0.2500,0.52,0.2704,0.18
2003-05-31,0.61,-0.26,-0.67,-1.860,0.02,8.22,24785.0,285.0,-0.47,0.0000,...,0.60,0.43,0.2500,0.50,0.2500,0.10,0.3025,0.55,0.3025,0.15


In [4]:
df.shape

(156, 92)

## Renomeação das colunas

Os nomes originais foram truncados em 30 caracteres, o que gerou colunas duplicadas (por exemplo, quatro colunas `FGV Brazil General Prices IGP-`). Nomes repetidos não podem ser diferenciados com `rename`, então a renomeação é feita **por posição**, usando o dicionário de dados em `references/data_dictionary.csv`.

Para cada coluna, o dicionário registra a variável correspondente no artigo (Tabelas 12 e 13), a transformação inferida (nível, variação % ou primeira diferença) e a confiança de cada decisão.

Passos:
1. Carregar o dicionário.
2. Separar a linha que existe só para documentação.
3. Conferir que a ordem das colunas do arquivo bate com o dicionário.
4. Renomear.
5. Conferir que nenhum dado foi perdido ou trocado de lugar.

`df` é uma cópia (`.copy()`), então `result['BRinf']` guarda os nomes originais para a conferência final.

In [5]:
dicionario = pd.read_csv(
    "../references/data_dictionary.csv",
    dtype={"position": "Int64", "paper_number": "Int64"},  # inteiros que aceitam vazio
)

print(dicionario.shape)
dicionario.head()

(93, 11)


,position,original_name,new_name,paper_number,paper_name,group,transformation,name_confidence,transform_confidence,in_dataset,notes
0,1,Brazil CPI IPCA MoM,ipca_mom,1,Brazil CPI IPCA,prices,pct_change_mom,high,high,yes,Target variable. Jan/2003 = 2.25 matches IBGE.
1,2,FGV Brazil General Prices IGP-,igpm_mom,2,FGV Brazil General Prices IGP-M,prices,pct_change_mom,high,high,yes,Jan/2003 = 2.33 matches the official IGP-M.
2,3,FGV Brazil General Prices IGP-,igpdi_mom,3,FGV Brazil General Prices IGP-DI,prices,pct_change_mom,high,high,yes,Jan/2003 = 2.17 matches the official IGP-DI.
3,4,FGV Brazil General Prices IGP-,fgv_igp_unknown,<NA>,NaN,prices,unknown,low,low,yes,Not identified. Corr 0.95 with IGP-DI but ~2.7...
4,5,FGV Brazil General Prices IGP-,igp10_mom,4,FGV Brazil General Prices IGP-10,prices,pct_change_mom,medium,high,yes,Magnitude close to IGP-M/IGP-DI. Truncated nam...


O dicionário tem **93 linhas**, uma a mais que as 92 colunas da base. A linha extra (`in_dataset == "no"`) é o **IPCA-15**: o artigo lista essa variável (nº 5 da Tabela 12), mas ela não existe no `BRinf.rda`, porque as posições 2 a 5 são todas índices da FGV. Ela fica no dicionário só como documentação, sem coluna de dados correspondente.

Por isso o filtro `in_dataset == "yes"` **não descarta nenhuma coluna da base**: ele só remove essa linha de documentação.

In [6]:
dicionario[dicionario["in_dataset"] == "no"]

,position,original_name,new_name,paper_number,paper_name,group,transformation,name_confidence,transform_confidence,in_dataset,notes
92,<NA>,NaN,NaN,5,Brazil CPI IPCA-15,prices,NaN,NaN,NaN,no,Documentation only: listed in the paper but no...


In [7]:
colunas = dicionario[dicionario["in_dataset"] == "yes"].sort_values("position")

assert len(colunas) == df.shape[1], "O dicionário não tem uma linha para cada coluna da base"
assert (colunas["original_name"].values == df.columns.values).all(), (
    "Nomes originais não batem com o dicionário (a renomeação já foi rodada?)"
)

df.columns = colunas["new_name"].values

with pd.option_context("display.max_rows", 100):
    display(colunas[["position", "original_name", "new_name", "name_confidence", "transformation"]])

,position,original_name,new_name,name_confidence,transformation
0,1,Brazil CPI IPCA MoM,ipca_mom,high,pct_change_mom
1,2,FGV Brazil General Prices IGP-,igpm_mom,high,pct_change_mom
2,3,FGV Brazil General Prices IGP-,igpdi_mom,high,pct_change_mom
3,4,FGV Brazil General Prices IGP-,fgv_igp_unknown,low,unknown
4,5,FGV Brazil General Prices IGP-,igp10_mom,medium,pct_change_mom
5,6,Brazil CPI IPCA Median Market,focus_ipca_12m_median,medium,level
6,7,Brazil Total Electricity Consu,electricity_consumption_total,high,level
7,8,Brazil Industrial Electricity,electricity_consumption_ind,medium,first_diff
8,9,BofA Merrill Lynch Economic Co,bofa_economic_condition_idx,medium,level
9,10,CNI Brazil Manufacture Industr,cni_capacity_util_diff,low,first_diff


In [8]:
duplicatas = df.columns.duplicated(keep=False)

df.columns[duplicatas]

Index([], dtype='str')

Conferência final: a base renomeada tem o mesmo formato e exatamente os mesmos valores da original, na mesma ordem. Só os nomes mudaram.

In [9]:
original = result["BRinf"]

assert df.shape == original.shape                    # mesmas 156 linhas e 92 colunas
assert (df.values == original.values).all()          # nenhum valor alterado ou fora de lugar
assert not df.columns.duplicated().any()             # nenhum nome repetido
assert df.columns.str.len().max() <= 30              # limite de 30 caracteres respeitado

df.shape

(156, 92)

In [10]:
df.head(5)

,ipca_mom,igpm_mom,igpdi_mom,fgv_igp_unknown,igp10_mom,focus_ipca_12m_median,electricity_consumption_total,electricity_consumption_ind,bofa_economic_condition_idx,cni_capacity_util_diff,...,focus_top5_med_t12,focus_top5_med_t13,focus_med_t1_sq,focus_mean_t1,focus_mean_t1_sq,focus_std_t1,focus_med_t2_sq,focus_mean_t2,focus_mean_t2_sq,focus_std_t2
2003-01-31,2.25,2.33,2.17,5.816,2.29,11.56,25422.0,-359.0,-1.47,0.0000,...,0.80,0.68,4.0000,1.96,3.8416,0.18,1.3225,1.15,1.3225,0.24
2003-02-28,1.57,2.28,1.59,4.351,2.42,10.95,25163.0,-303.0,-1.21,80.5999,...,0.68,0.48,2.5600,1.60,2.5600,0.26,0.7569,0.91,0.8281,0.27
2003-03-31,1.23,1.53,1.66,4.593,1.58,9.58,24960.0,115.0,-0.35,-0.3000,...,0.55,0.50,0.8836,0.93,0.8649,0.13,0.4900,0.73,0.5329,0.17
2003-04-30,0.97,0.92,0.41,1.173,1.24,8.88,25284.0,93.0,-0.15,-0.5000,...,0.45,0.78,1.0000,0.97,0.9409,0.14,0.2500,0.52,0.2704,0.18
2003-05-31,0.61,-0.26,-0.67,-1.860,0.02,8.22,24785.0,285.0,-0.47,0.0000,...,0.60,0.43,0.2500,0.50,0.2500,0.10,0.3025,0.55,0.3025,0.15


In [11]:
df.index = pd.to_datetime(df.index)
df.index.dtype

dtype('<M8[us]')

In [12]:
df.loc["2011-04", "electricity_consumption_total"] = df.loc["2011-03-31","electricity_consumption_total"]

In [13]:
df.loc["2011-04", "electricity_consumption_total"]

2011-04-30    35533.0
Name: electricity_consumption_total, dtype: float64

In [14]:
df.loc[(df.index == "2003-01-31") | (df.index == "2003-02-28")]

,ipca_mom,igpm_mom,igpdi_mom,fgv_igp_unknown,igp10_mom,focus_ipca_12m_median,electricity_consumption_total,electricity_consumption_ind,bofa_economic_condition_idx,cni_capacity_util_diff,...,focus_top5_med_t12,focus_top5_med_t13,focus_med_t1_sq,focus_mean_t1,focus_mean_t1_sq,focus_std_t1,focus_med_t2_sq,focus_mean_t2,focus_mean_t2_sq,focus_std_t2
2003-01-31,2.25,2.33,2.17,5.816,2.29,11.56,25422.0,-359.0,-1.47,0.0000,...,0.80,0.68,4.00,1.96,3.8416,0.18,1.3225,1.15,1.3225,0.24
2003-02-28,1.57,2.28,1.59,4.351,2.42,10.95,25163.0,-303.0,-1.21,80.5999,...,0.68,0.48,2.56,1.60,2.5600,0.26,0.7569,0.91,0.8281,0.27


In [15]:
df = df.drop(["2003-01-31", "2003-02-28"])

In [16]:
df.head(3)

,ipca_mom,igpm_mom,igpdi_mom,fgv_igp_unknown,igp10_mom,focus_ipca_12m_median,electricity_consumption_total,electricity_consumption_ind,bofa_economic_condition_idx,cni_capacity_util_diff,...,focus_top5_med_t12,focus_top5_med_t13,focus_med_t1_sq,focus_mean_t1,focus_mean_t1_sq,focus_std_t1,focus_med_t2_sq,focus_mean_t2,focus_mean_t2_sq,focus_std_t2
2003-03-31,1.23,1.53,1.66,4.593,1.58,9.58,24960.0,115.0,-0.35,-0.3,...,0.55,0.50,0.8836,0.93,0.8649,0.13,0.4900,0.73,0.5329,0.17
2003-04-30,0.97,0.92,0.41,1.173,1.24,8.88,25284.0,93.0,-0.15,-0.5,...,0.45,0.78,1.0000,0.97,0.9409,0.14,0.2500,0.52,0.2704,0.18
2003-05-31,0.61,-0.26,-0.67,-1.860,0.02,8.22,24785.0,285.0,-0.47,0.0,...,0.60,0.43,0.2500,0.50,0.2500,0.10,0.3025,0.55,0.3025,0.15


In [18]:
df.to_csv("../data/processed/BRinf_clean.csv")